In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
# Установка необходимых библиотек
!pip install owlready2 pandas -q

print("✅ Библиотеки установлены")

# Импорт библиотек
from owlready2 import *
import pandas as pd

print("✅ Библиотеки импортированы")

✅ Библиотеки установлены
✅ Библиотеки импортированы


In [52]:
from owlready2 import *
import pandas as pd

users_df = pd.read_csv("/content/drive/MyDrive/data_sets/users.csv")
posts_df = pd.read_csv("/content/drive/MyDrive/data_sets/posts.csv")
publishes_df = pd.read_csv("/content/drive/MyDrive/data_sets/publishes.csv")

print(len(users_df), "users |", len(posts_df), "posts")


11 users | 11 posts


Задание 1-5

In [53]:
# Создаём онтологию
onto = get_ontology("http://example.org/social_ai.owl")

with onto:
    # Классы
    class User(Thing): pass
    class Post(Thing): pass

    # Data свойства
    class has_name(User >> str, FunctionalProperty): pass
    class has_title(Post >> str, FunctionalProperty): pass

    # Object свойства
    class has_published(User >> Post): pass

# Создаём экземпляры пользователей
users = {}
with onto:
    for _, row in users_df.iterrows():
        u = User(f"user_{row.id}")
        u.has_name = row.name
        users[row.id] = u

# Создаём экземпляры постов
posts = {}
with onto:
    for _, row in posts_df.iterrows():
        p = Post(f"post_{row.id}")
        p.has_title = row.title
        posts[row.id] = p

# Связи has_published
for _, row in publishes_df.iterrows():
    u = users.get(row.user_id)
    p = posts.get(row.post_id)
    if u and p:
        u.has_published.append(p)

# Проверка
print("Проверка пользователей и их постов:")
for u in list(onto.User.instances())[:5]:
    print(u.has_name, "→", [p.has_title[0] for p in u.has_published])


Проверка пользователей и их постов:
['Алексей', 'Алексей', 'Алексей'] → ['Новый ИИ от OpenAI способен писать музыку', 'Новый ИИ от OpenAI способен писать музыку']
['Мария', 'Мария', 'Мария'] → ['Илон Маск планирует имплантировать чипы в мозг', 'Илон Маск планирует имплантировать чипы в мозг']
['Иван', 'Иван', 'Иван'] → ['Почему стоит опасаться глубоких фейков', 'Почему стоит опасаться глубоких фейков']
['Екатерина', 'Екатерина', 'Екатерина'] → ['ИИ ошибся и выдал фейковую новость', 'ИИ ошибся и выдал фейковую новость']
['Николай', 'Николай', 'Николай'] → ['ИИ улучшает качество фото с низким разрешением', 'ИИ улучшает качество фото с низким разрешением']


Задание 6-9

In [54]:
# Добавляем класс Category и подклассы
with onto:
    class Category(Thing): pass
    class FakeNews(Category): pass
    class Opinion(Category): pass
    class Fact(Category): pass
    class Meme(Category): pass
    class Educational(Category): pass

    # Свойство Post → Category
    class has_category(Post >> Category, FunctionalProperty): pass

# Создаём категории из CSV
categories = {}
with onto:
    for _, row in categories_df.iterrows():
        cat = Category(row.name)
        categories[row.name] = cat

# Связываем посты с категориями (FunctionalProperty — 1 категория на пост)
for _, row in posts_df.iterrows():
    p = posts.get(row.id)
    cat_name = row.get('category', None)
    if p and cat_name in categories:
        p.has_category = categories[cat_name]

print("\nПроверка пользователей и постов с категориями:")
for u in list(onto.User.instances())[:5]:
    # has_name тоже может быть списком
    user_name = u.has_name[0] if u.has_name else "None"
    print(f"👤 {user_name}")
    for p in u.has_published:
        title = p.has_title[0] if p.has_title else "None"
        cat = p.has_category[0].name if p.has_category else "None"
        print(f"   📄 {title} → {cat}")



Проверка пользователей и постов с категориями:
👤 Алексей
   📄 Новый ИИ от OpenAI способен писать музыку → ai
   📄 Новый ИИ от OpenAI способен писать музыку → ai
👤 Мария
   📄 Илон Маск планирует имплантировать чипы в мозг → other
   📄 Илон Маск планирует имплантировать чипы в мозг → other
👤 Иван
   📄 Почему стоит опасаться глубоких фейков → other
   📄 Почему стоит опасаться глубоких фейков → other
👤 Екатерина
   📄 ИИ ошибся и выдал фейковую новость → None
   📄 ИИ ошибся и выдал фейковую новость → None
👤 Николай
   📄 ИИ улучшает качество фото с низким разрешением → None
   📄 ИИ улучшает качество фото с низким разрешением → None


Задание 9-14

In [55]:
# Загружаем онтологию
onto = get_ontology("http://example.org/social_ai.owl")
onto.load(file="/content/drive/MyDrive/data_sets/social_ai_v2.owl")

with onto:
    # Задание 10: аксиома "каждый пост имеет категорию"
    # В OWL это можно записать через классовое ограничение
    class PostWithCategory(Post):
        equivalent_to = [Post & (has_category.some(Category))]

    # Задание 12: подкласс FakeNewsPost
    class FakeNewsPost(Post): pass

# Задание 13: применяем подкласс FakeNewsPost к постам с категорией 'FakeNews'
for p in onto.Post.instances():
    if p.has_category and p.has_category[0].name == "FakeNews":
        p.is_a.append(FakeNewsPost)

# Задание 14: проверка FakeNewsPost
print("Посты типа FakeNewsPost:")
for p in onto.FakeNewsPost.instances():
    print(f"📄 {p.has_title[0]}")


Посты типа FakeNewsPost:


Задание 15-20

In [56]:
from owlready2 import *

# Загружаем онтологию
onto = get_ontology("http://example.org/social_ai.owl")
onto.load(file="/content/drive/MyDrive/data_sets/social_ai_v2.owl")

# ------------------------------
# Итерация 3: Аксиомы и подклассы
# ------------------------------

with onto:
    # Задание 10: Аксиома — каждый пост имеет хотя бы одну категорию
    class PostWithCategory(Post):
        equivalent_to = [Post & (has_category.some(Category))]

    # Задание 12: подкласс FakeNewsPost
    class FakeNewsPost(Post): pass

# Применяем FakeNewsPost к постам с категорией 'FakeNews'
for p in onto.Post.instances():
    if p.has_category:
        cat_name = p.has_category[0].name.strip().lower()
        if cat_name == "fakenews":
            if FakeNewsPost not in p.is_a:
                p.is_a.append(FakeNewsPost)

# Проверка: посты типа FakeNewsPost
print("Посты типа FakeNewsPost:")
for p in onto.FakeNewsPost.instances():
    print(f"📄 {p.has_title[0]} → {p.has_category[0].name}")

# ------------------------------
# Итерация 4: Инверсивное отношение и LegalCase
# ------------------------------

with onto:
    # Задание 15: инверсивное отношение has_author
    class has_author(Post >> User, InverseFunctionalProperty):
        inverse_property = has_published

    # Задание 15: LegalCase
    class LegalCase(Thing): pass

    # Задание 16: involved_in_case
    if not hasattr(User, "involved_in_case"):
        class involved_in_case(User >> LegalCase): pass

# Задание 17: эмуляция SWRL правила вручную
for u in onto.User.instances():
    for p in u.has_published:
        if isinstance(p, FakeNewsPost):
            case_name = f"case_{u.name}_{p.name}"
            case = LegalCase(case_name)
            u.involved_in_case.append(case)

# Проверка: кто участвует в LegalCase
print("\nПользователи, участвующие в LegalCase:")
for u in onto.User.instances():
    if hasattr(u, "involved_in_case") and u.involved_in_case:
        cases = [c.name for c in u.involved_in_case]
        user_name = u.has_name[0] if u.has_name else u.name
        print(f"👤 {user_name} → {cases}")

# ------------------------------
# Итерация 4-5: Сохранение онтологии
# ------------------------------

onto.save(file="/content/drive/MyDrive/data_sets/social_ai_final.owl", format="rdfxml")
print("\nОнтология сохранена как social_ai_final.owl")


Посты типа FakeNewsPost:

Пользователи, участвующие в LegalCase:

Онтология сохранена как social_ai_final.owl


In [57]:
# Задание 17: если User публикует FakeNewsPost → участвует в LegalCase
for u in onto.User.instances():
    for p in u.has_published:
        if isinstance(p, FakeNewsPost):
            # создаём отдельный LegalCase для каждого случая
            case = LegalCase(f"case_{u.name}_{p.name}")
            u.involved_in_case.append(case)

# Проверка: кто участвует в LegalCase
print("\nПользователи, участвующие в LegalCase:")
for u in onto.User.instances():
    if hasattr(u, "involved_in_case") and u.involved_in_case:
        cases = [c.name for c in u.involved_in_case]
        print(f"👤 {u.has_name[0]} → {cases}")



Пользователи, участвующие в LegalCase:


In [58]:
onto.save(file="/content/drive/MyDrive/data_sets/social_ai_final.owl", format="rdfxml")
print("Онтология сохранена как social_ai_final.owl")


Онтология сохранена как social_ai_final.owl


In [59]:
import networkx as nx
import matplotlib.pyplot as plt

# Получаем RDF-граф через rdflib
g = onto.world.as_rdflib_graph()
G = nx.DiGraph()

# Добавляем ребра: s → o с подписью свойства
for s, p, o in g:
    s_str = str(s).split("#")[-1]
    o_str = str(o).split("#")[-1]
    p_str = str(p).split("#")[-1]
    G.add_edge(s_str, o_str, label=p_str)

# Настройка графа
plt.figure(figsize=(20, 12))
pos = nx.spring_layout(G, k=0.5, iterations=100)

# Рисуем узлы и подписи
nx.draw(G, pos, with_labels=True, node_size=800, node_color="#87CEFA", font_size=10, font_weight="bold", arrows=True)
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, label_pos=0.5)

plt.title("Онтологический граф социальной сети", fontsize=16)
plt.axis("off")
plt.show()


ModuleNotFoundError: No module named 'rdflib'